[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matth426/Calculus-with-Python-Programming/blob/main/notebooks/calcwp_wk8_motion_problems.ipynb)

# CALCWP: CALCULUS WITH PYTHON PROGRAMMING
## Week 8 — Rates of Change and Motion Problems

**Topics this week:**
- Building a reusable "motion analyzer" using `def`
- Determining direction of motion and when an object is at rest
- Speeding up vs. slowing down (comparing signs of velocity and acceleration)
- Distance traveled vs. displacement

**By the end of this notebook, you should be able to:**
1. Write a reusable Python function that computes velocity and acceleration from a position function.
2. Determine when a moving object is at rest, moving forward, or moving backward.
3. Determine when an object is speeding up or slowing down.
4. Distinguish between total distance traveled and net displacement, and compute both.

---


## 0. Quick Recap from Week 7

Last week, we saw that differentiating position twice gives acceleration: $ s(t) \to v(t) = s'(t) \to a(t) = v'(t) $. This week, we put that chain to work on full motion problems — the kind of questions a physics or engineering course would ask, but solved with the tools we've built. We'll also write our first genuinely reusable "analysis" function, one that does several calculus steps at once.


## 1. Building a Reusable Motion Analyzer

Every motion problem starts the same way: given a position function $s(t)$, find $v(t)$ and $a(t)$. Instead of retyping `diff()` calls every time, let's write **one function** that does this for any position function we give it.


In [ ]:
from sympy import symbols, diff, solve

t = symbols('t')

def analyze_motion(s):
    """Given a position function s(t), return velocity and acceleration functions."""
    v = diff(s, t)
    a = diff(v, t)
    return v, a

Let's test it on the falling-ball example from Weeks 5 and 7: $ h(t) = -5t^2 + 20t $.

In [ ]:
h = -5*t**2 + 20*t

velocity, acceleration = analyze_motion(h)
print("Position:     h(t) =", h)
print("Velocity:     v(t) =", velocity)
print("Acceleration: a(t) =", acceleration)

Notice this function returns **two values at once** using `return v, a` — this is called returning a **tuple**. When we call it, we can "unpack" both results into two variables in one line: `velocity, acceleration = analyze_motion(h)`.

**Practice 1.1:** Use `analyze_motion()` to find the velocity and acceleration for a new position function $ s(t) = t^3 - 9t^2 + 24t $.

In [ ]:
# Write your code here


<details>
<summary><b>Click for Solution</b></summary>

```python
s = t**3 - 9*t**2 + 24*t
v, a = analyze_motion(s)
print("v(t) =", v)
print("a(t) =", a)
```

**Answers:** $v(t) = 3t^2 - 18t + 24$, $a(t) = 6t - 18$

</details>

## 2. Direction of Motion

The **sign** of velocity tells us the direction of motion:
- $ v(t) > 0 $: moving in the positive direction (forward / upward, depending on context)
- $ v(t) < 0 $: moving in the negative direction (backward / downward)
- $ v(t) = 0 $: momentarily at rest

To find when an object is at rest, we solve $v(t) = 0$.

Using $ s(t) = t^3 - 9t^2 + 24t $ from Practice 1.1:


In [ ]:
s = t**3 - 9*t**2 + 24*t
v, a = analyze_motion(s)

rest_times = solve(v, t)
print("v(t) =", v)
print("Object is at rest when t =", rest_times)

So the object is at rest at $t=2$ and $t=4$. Between, before, and after these times, it must be moving in one direction or the other — let's check by evaluating $v(t)$ at a test point in each interval.


In [ ]:
test_points = [0, 3, 5]   # one point before t=2, one between 2 and 4, one after t=4

for tp in test_points:
    v_value = v.subs(t, tp)
    direction = "forward (v > 0)" if v_value > 0 else ("backward (v < 0)" if v_value < 0 else "at rest (v = 0)")
    print(f"At t={tp}: v(t) = {v_value}  -->  {direction}")

**Practice 2.1:** For the same $s(t) = t^3 - 9t^2 + 24t$, we found rest points at $t=2$ and $t=4$. Using the test points already shown above, describe in words what the object does over the full interval from $t=0$ to $t=5$ (forward, then backward, then forward again? or some other pattern?).

In [ ]:
# Write your comment/description here based on the results above


<details>
<summary><b>Click for Solution</b></summary>

```python
# Based on the test points:
# t=0: v(0) = 24  --> moving forward
# t=3: v(3) = -3  --> moving backward
# t=5: v(5) = 9   --> moving forward
#
# So the object moves FORWARD from t=0 to t=2, reverses and moves BACKWARD
# from t=2 to t=4, then moves FORWARD again from t=4 onward.
```

</details>

## 3. Speeding Up or Slowing Down?

This is a common source of confusion: **speed increasing** is not the same as **velocity increasing**. An object can have negative velocity (moving backward) while still *speeding up* — if it's accelerating further in the negative direction.

The rule: compare the **signs** of velocity and acceleration at a given time.

- If velocity and acceleration have the **same sign** (both positive or both negative): the object is **speeding up**.
- If velocity and acceleration have **opposite signs**: the object is **slowing down**.

Let's check this for our object at a few times.


In [ ]:
def describe_motion(v_func, a_func, time):
    v_value = v_func.subs(t, time)
    a_value = a_func.subs(t, time)

    if v_value == 0:
        speed_status = "momentarily at rest"
    elif (v_value > 0 and a_value > 0) or (v_value < 0 and a_value < 0):
        speed_status = "speeding up"
    elif (v_value > 0 and a_value < 0) or (v_value < 0 and a_value > 0):
        speed_status = "slowing down"
    else:
        speed_status = "constant speed (a = 0)"

    print(f"t={time}: v={v_value}, a={a_value}  -->  {speed_status}")

for tp in [0, 1, 3, 4.5]:
    describe_motion(v, a, tp)

Notice `describe_motion()` is a `def` function that takes **two other functions** as input (`v_func` and `a_func`), plus a time — combining ideas from this week and Week 5 (where `difference_quotient` also took a function as an argument).

**Practice 3.1:** Using the `h(t) = -5t^2 + 20t` ball example from Section 1, use `describe_motion()` to check whether the ball is speeding up or slowing down at $t=1$ and at $t=3$.

In [ ]:
# Write your code here


<details>
<summary><b>Click for Solution</b></summary>

```python
h_velocity, h_acceleration = analyze_motion(h)

describe_motion(h_velocity, h_acceleration, 1)
describe_motion(h_velocity, h_acceleration, 3)
```

**Interpretation:** At $t=1$, the ball is still rising (positive velocity) but gravity is decelerating it — slowing down. At $t=3$, the ball is falling (negative velocity) and gravity keeps pulling it faster downward — speeding up.

</details>

## 4. Total Distance Traveled vs. Displacement

**Displacement** is simply how far the final position is from the starting position — it can be positive, negative, or zero, and it ignores any back-and-forth movement along the way.

**Total distance traveled** adds up *every bit of movement*, regardless of direction — it can never decrease and is always $\geq 0$.

If an object changes direction during a time interval (i.e., $v(t) = 0$ somewhere inside it), these two values will differ.

**Method:** Break the interval into pieces at each rest point, compute the *absolute* distance covered in each piece, then add them all up.

Using $ s(t) = t^3 - 9t^2 + 24t $ over $[0, 5]$, with rest points at $t=2$ and $t=4$:


In [ ]:
s = t**3 - 9*t**2 + 24*t

# Evaluate position at the start, each rest point, and the end
positions = {
    0: s.subs(t, 0),
    2: s.subs(t, 2),
    4: s.subs(t, 4),
    5: s.subs(t, 5),
}

for time, pos in positions.items():
    print(f"s({time}) = {pos}")

In [ ]:
# Displacement: just the difference between final and initial position
displacement = positions[5] - positions[0]
print("Displacement (t=0 to t=5):", displacement)

# Total distance: sum of absolute distances between consecutive rest points
leg1 = abs(positions[2] - positions[0])   # t=0 to t=2
leg2 = abs(positions[4] - positions[2])   # t=2 to t=4
leg3 = abs(positions[5] - positions[4])   # t=4 to t=5

total_distance = leg1 + leg2 + leg3
print("Total distance traveled:", total_distance)

Notice the total distance traveled is larger than the displacement — because the object reversed direction twice, and every reversal adds "wasted" distance that displacement doesn't count.

**Practice 4.1:** For the ball $ h(t) = -5t^2 + 20t $, the rest point occurs at $t=2$ (found in Practice 3.1's context). Compute both the displacement and total distance traveled over the interval $[0, 3]$.

In [ ]:
# Write your code here


<details>
<summary><b>Click for Solution</b></summary>

```python
h_positions = {
    0: h.subs(t, 0),
    2: h.subs(t, 2),   # the rest point
    3: h.subs(t, 3),
}
for time, pos in h_positions.items():
    print(f"h({time}) = {pos}")

displacement = h_positions[3] - h_positions[0]
total_distance = abs(h_positions[2] - h_positions[0]) + abs(h_positions[3] - h_positions[2])

print("Displacement:", displacement)
print("Total distance traveled:", total_distance)
```

**Interpretation:** The ball rises to its peak (t=2) and falls partway back down by t=3. Total distance traveled is larger than displacement because the ball backtracked after reaching its peak.

</details>

## 5. Visualizing All Three Together

A common way to understand motion is to plot position, velocity, and acceleration on stacked graphs sharing the same time axis, so you can see how they relate.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import lambdify

s_expr = t**3 - 9*t**2 + 24*t
v_expr, a_expr = analyze_motion(s_expr)

# Convert sympy expressions into fast numeric functions for plotting
s_func = lambdify(t, s_expr, 'numpy')
v_func = lambdify(t, v_expr, 'numpy')
a_func = lambdify(t, a_expr, 'numpy')

t_vals = np.linspace(0, 5, 200)

fig, axes = plt.subplots(3, 1, figsize=(7, 9), sharex=True)

axes[0].plot(t_vals, s_func(t_vals), color='steelblue')
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].set_ylabel("s(t)\n(position)")
axes[0].set_title("Position, Velocity, and Acceleration Over Time")
axes[0].grid(True)

axes[1].plot(t_vals, v_func(t_vals), color='orange')
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].set_ylabel("v(t)\n(velocity)")
axes[1].grid(True)

axes[2].plot(t_vals, a_func(t_vals), color='green')
axes[2].axhline(0, color='gray', linewidth=0.5)
axes[2].set_ylabel("a(t)\n(acceleration)")
axes[2].set_xlabel("t")
axes[2].grid(True)

plt.tight_layout()
plt.show()

Notice: wherever the velocity graph (middle) crosses zero, the position graph (top) has a peak or a valley — those are the rest points we found algebraically in Section 2. This is a nice visual confirmation that our `solve(v, t)` approach lines up with the graph.

**New tool: `lambdify`.** We used `sympy.lambdify()` to convert a symbolic expression into a fast numeric Python function that works with `numpy` arrays — this is the standard way to go from "exact symbolic math" (Weeks 4–7) back to "numeric graphing" (Weeks 1–3) when you need to plot something.


## 6. Mini-Challenge

An elevator's height above the ground (in meters), starting at $t=0$, is modeled by:

$$ s(t) = -t^3 + 6t^2 $$

for $0 \le t \le 6$ seconds.

**Tasks:**
1. Use `analyze_motion()` to find $v(t)$ and $a(t)$.
2. Find all times when the elevator is at rest.
3. Determine the direction of motion (up or down) just before and just after each rest point, using test values.
4. Compute the total distance traveled by the elevator from $t=0$ to $t=6$.
5. **Bonus:** Plot position, velocity, and acceleration (like Section 5) to visually confirm your answers.


In [ ]:
# Task 1: Get v(t) and a(t)


# Task 2: Find rest points


# Task 3: Determine direction before/after each rest point (test points + comments)


# Task 4: Total distance traveled from t=0 to t=6


# Task 5 (Bonus): Plot all three


<details>
<summary><b>Click for Solution</b></summary>

```python
s = -t**3 + 6*t**2

v, a = analyze_motion(s)
print("v(t) =", v)
print("a(t) =", a)

rest_times = solve(v, t)
print("Rest times:", rest_times)

# Test direction around the rest point(s)
for tp in [0, 3, 5]:
    v_value = v.subs(t, tp)
    print(f"t={tp}: v={v_value}")

# Total distance: break at rest point(s) within [0, 6]
checkpoints = sorted(set([0] + rest_times + [6]))
positions = {tm: s.subs(t, tm) for tm in checkpoints}

total_distance = sum(
    abs(positions[checkpoints[i+1]] - positions[checkpoints[i]])
    for i in range(len(checkpoints) - 1)
)
print("Positions:", positions)
print("Total distance traveled:", total_distance)
```

**Answers:** $v(t) = -3t^2 + 12t$, $a(t) = -6t + 12$. Rest points at $t=0$ and $t=4$. The elevator moves up from $t=0$ to $t=4$, then down from $t=4$ to $t=6$.

</details>

## Summary

This week, you learned:
- How to build a reusable `def` function (`analyze_motion`) that returns multiple values at once (a tuple)
- How to interpret the sign of velocity as direction, and find rest points by solving $v(t)=0$
- How to determine speeding up vs. slowing down by comparing the signs of velocity and acceleration
- The difference between displacement (net change in position) and total distance traveled (sum of all movement)
- How to use `sympy.lambdify()` to convert a symbolic expression into a numeric function for plotting, and how position/velocity/acceleration graphs relate to each other visually

**Next week (Month 3 begins!):** We use derivatives to **sketch curves** — identifying increasing/decreasing behavior, concavity, and inflection points, tying together nearly everything from Weeks 5–8.

---
*CALCWP — Calculus with Python Programming | Mattheus Marcus Contreras | Lab Manual Series*
